# Notebook 06: test evaluation, 2024 (Section 10.1, SOP 4)

**The 2024 test season is opened once, here, after G17 and the `settings-frozen` tag.** Start this deliberately.
G18 checks that the tag is an ancestor of HEAD, that every frozen file still has the SHA-256 recorded in
`configs/frozen_manifest.json`, and that the test has not been opened before (`gates/TEST_UNSEALED.json`).
A second opening needs a written reason (`phase6.run(rerun_reason=...)`), which is appended to the marker and must
be reported in `reports/transparency_log.md`. Nothing in this notebook may change a frozen setting. Results are
reported whatever they are.

Same as `python scripts/run_phase.py --phase 6`. Dry run: set `PASIG_PROFILE=rehearsal` (the 2023 validation
windows stand in for 2024, which is never read; outputs under `runs/rehearsal/`; the numbers mean nothing).

**Runs locally:** the raw data are committed in `data/raw/`; the first cell checks them against the manifest and, in a fresh clone, rebuilds any missing generated inputs (no Google Drive needed).

In [ ]:
import os, sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
from pasig.local import prepare
prepare(6)   # raw data from the repo (no Drive); rebuilds missing local inputs once per clone
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '-1')   # inference on CPU: reproducible scores
# os.environ['PASIG_PROFILE'] = 'rehearsal'          # uncomment for the dry run
import pandas as pd
from IPython.display import Image, display
from pasig import phase6
from pasig.config import out_dir, load, profile
from pasig.gates import require_gates, read_gate
require_gates('G17')
T, F = out_dir('tables'), out_dir('figures')
profile()

## 1. G18: unseal the test season (blocking)

In [ ]:
phase6.run(['unseal'])
g = read_gate('G18'); g['decision'], g['evidence']['manifest']['n_files'], g['evidence']['tag_commit']

## 2. Test windows (frozen builder, tide, scalers) and predictions of every model

In [ ]:
phase6.run(['predict'])
json.loads((T / 'test_windows_summary.json').read_text())

## 3. Metrics, alerts, equal-recall false alarms, events, Diebold–Mariano + Holm, bootstrap CIs

In [ ]:
res = phase6.run(['evaluate'])['evaluate']
{k: res[k] for k in ('stand_in', 'test_seeds', 'strongest_simple_baseline', 'classifier_input_sets')}

### Level forecasts: RMSE by lead (neural: mean over 5 seeds; SD in the CSV) and skill

In [ ]:
lv = res['tables']['level']
lv[lv.subset.eq('all')].pivot(index='model', columns='lead', values='rmse').round(4)

In [ ]:
lv[lv.subset.eq('rainy')].pivot(index='model', columns='lead', values='rmse').round(4)

In [ ]:
sk = res['tables']['skill']
sk[sk.subset.eq('all') & sk.model.eq('hybrid')].round(3)

In [ ]:
pd.read_csv(T / 'test_rmse_diff_ci.csv').round(4)   # RMSE(hybrid) - RMSE(reference), moving-block 95% CI

In [ ]:
display(Image(F / 'test_rmse_by_lead.png')); display(Image(F / 'test_series_lead3.png'))

### H3: Diebold–Mariano (HLN, Newey–West), Holm over 72 tests

In [ ]:
dm = res['tables']['dm']
dm.pivot_table(index=['subset', 'baseline'], columns='lead', values='p_holm').round(4)

In [ ]:
res['H3']

### Hourly alerts with the frozen trigger offsets and classifier thresholds (lead 3 shown; all leads in the CSV)

In [ ]:
al = res['tables']['alerts']
al[al.subset.eq('all') & al.lead.eq(3)][['model', 'level', 'offset_m', 'tp', 'fp', 'fn', 'precision', 'recall', 'f1']].round(3)

### H4: false alarms at Recall ≥ 0.90 and PR-AUC, hybrid vs classifier families (high water decides; D19)

In [ ]:
er = res['tables']['equal_recall']
cols = ['level', 'family', 'lead', 'positives', 'fp_hybrid_r90', 'fp_clf_r90', 'fp_diff_r90', 'fp_diff_r90_lo',
        'fp_diff_r90_hi', 'recall_hybrid_r90', 'recall_clf_r90', 'pr_auc_hybrid', 'pr_auc_clf', 'pr_auc_diff_lo', 'pr_auc_diff_hi']
er[er.subset.eq('all')][[c for c in cols if c in er]].round(3)

In [ ]:
res['H4']

### Events: detection rate (event-bootstrap 95% CI) and median warning time

In [ ]:
res['tables']['events'].groupby('level').agg(events=('event', 'size'), median_hours=('hours', 'median'), max_peak=('peak_m', 'max'))

In [ ]:
d = res['tables']['detection']
d[d.seed.isin([-1, *res['test_seeds'].values()])].pivot_table(index=['level', 'model'], columns='lead', values='detection_rate').round(2)

## Tests

In [ ]:
import subprocess
assert subprocess.call([sys.executable, '-m', 'pytest', '-q', 'tests/test_metrics.py', 'tests/test_events.py',
                        'tests/test_dm.py', 'tests/test_unseal.py', 'tests/test_same_hours.py']) == 0

Write the verdicts and every deviation into `reports/transparency_log.md` and `docs/decisions.md`; then commit
`gates/G18_*.json`, `gates/TEST_UNSEALED.json` and `reports/tables/test_*`, `reports/figures/test_*`.